In [1]:
import json
import numpy as np
import pandas as pd

In [3]:
# 1. Cargar la matriz desde el JSON
with open("../datos/resultados_hmm.json", "r", encoding="utf-8") as f:
    datos = json.load(f)

A = np.array(datos["matriz_transicion"])
estados = datos["estados"]

In [4]:
# 2. Construir la Matriz Absorbente P fijando E2 (índice 2) como estado absorbente
P_abs = A.copy()
P_abs[2, :] = [0.0, 0.0, 1.0, 0.0]

# Reordenamos las filas/columnas para dejar los estados transitorios arriba (E0, E1, E3)
# y el absorbente abajo (E2): forma canónica P = [[Q, R], [0, I]]
indices_transitorios = [0, 1, 3]
nombres_transitorios = ["E0 (Reconnaissance)", "E1 (Initial Access)", "E3 (Pivote)"]

Q = P_abs[np.ix_(indices_transitorios, indices_transitorios)]
R = P_abs[np.ix_(indices_transitorios, [2])]

In [5]:
# 3. Calcular la Matriz Fundamental N = (I - Q)^(-1)
I = np.eye(len(Q))
N = np.linalg.inv(I - Q)

df_N = pd.DataFrame(N, index=nombres_transitorios, columns=nombres_transitorios)
print("--- Matriz Fundamental N (Visitas esperadas a cada estado antes de Exfiltración) ---")
print(df_N.round(2))

--- Matriz Fundamental N (Visitas esperadas a cada estado antes de Exfiltración) ---
                     E0 (Reconnaissance)  E1 (Initial Access)  E3 (Pivote)
E0 (Reconnaissance)               168.90                24.23        18.80
E1 (Initial Access)               159.48                43.47        21.73
E3 (Pivote)                       164.98                28.93        23.25


In [6]:
# 4. Calcular Tiempos Medios de Absorción (t = N * 1)
tiempos_absorcion = N.sum(axis=1)
df_tiempos = pd.DataFrame({
    "Estado Inicial": nombres_transitorios,
    "Flujos Esperados hasta Exfiltración (E2)": tiempos_absorcion.round(2)
})
print("\n--- Tiempos Medios de Absorción ---")
print(df_tiempos.to_string(index=False))


--- Tiempos Medios de Absorción ---
     Estado Inicial  Flujos Esperados hasta Exfiltración (E2)
E0 (Reconnaissance)                                    211.93
E1 (Initial Access)                                    224.68
        E3 (Pivote)                                    217.16


In [11]:
df_original = pd.read_csv("../datos_procesados/original_ordenado.csv")